## 1. Setup & Configuration


In [ ]:
import os
import json
import numpy as np
import time
import warnings
import re
import pandas as pd
import tiktoken
import matplotlib.pyplot as plt
import seaborn as sns
import ollama
import copy
from sklearn.metrics.pairwise import cosine_similarity
from matplotlib.patches import FancyBboxPatch
from collections import Counter
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
from sentence_transformers import CrossEncoder


In [ ]:
warnings.filterwarnings("ignore")

# -- Load environment --
load_dotenv(dotenv_path=r"..\.env", override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key or "YOUR_" in api_key:
    raise ValueError("Invalid or missing OPENAI_API_KEY in .env")

# -- Initialize clients --
openai_client = OpenAI(api_key=api_key)
qdrant_client = QdrantClient(path = "qdrantdb")

## 2. Constants & Helper Functions


In [ ]:
# ── Constants ──
QUERY_FILE = r"..\queries_500.jsonl"
NUM_QUERIES = 500
BATCH_SIZE = 3          # queries per augmentation prompt
EMBED_MODEL = "text-embedding-3-large"
EMBED_DIM = 3072
RETRIEVAL_K = 5        

# Collection names (one per strategy)
COLLECTIONS = {
    "semantic":  "legal_semantic",
    "fixed":     "legal_fixed",
    "recursive": "legal_recursive",
}

# ── Tokenizer ──
encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    """Count tokens using the cl100k_base tokenizer."""
    return len(encoding.encode(text))

# ── Embedding function (same as rag.ipynb) ──
def embed_texts(texts):
    """Embed a list of texts using OpenAI text-embedding-3-large."""
    response = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in response.data]


## 5. Retrieval Function



In [ ]:
def retrieve_context(query, collection_name, k=RETRIEVAL_K):
    """Retrieve top-k relevant chunks from a Qdrant collection.
    
    Args:
        query: The search query text
        collection_name: Which Qdrant collection to search
        k: Number of results to return
    
    Returns:
        Formatted string of retrieved contexts with source metadata
    """
    query_embedding = embed_texts([query])[0]
    
    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        with_payload=True,
        limit=k
    )
    
    contexts = []
    for result in results.points:
        payload = result.payload
        contexts.append(
            f"[File: {payload.get('file', 'N/A')} | Section: {payload.get('section', 'N/A')}]\n"
            f"{payload.get('text', '')}"
        )
    
    return "\n\n".join(contexts)

## 6. Augmentation Prompt


In [ ]:
def build_batch_prompt(instructions, responses, collection_name):
    """Build the augmentation prompt for a batch of queries.
    
    This is the same prompt structure as rag.ipynb:
    - Strict legal assistant role
    - Per-instruction context retrieval
    - Citation and source rules
    - JSON output format
    
    Args:
        instructions: List of query strings (batch of 5)
        responses: List of original response strings
        collection_name: Which Qdrant collection to retrieve from
    
    Returns:
        Complete prompt string ready for LLM generation
    """
    context_sections = []
    instruction_lines = []
    response_lines = []
    
    for idx, (instruction, response) in enumerate(zip(instructions, responses), 1):
        # Retrieve context per instruction from the specified collection
        context = retrieve_context(instruction, collection_name)
        context_sections.append(f"[Instruction {idx} Context]\n{context}")
        instruction_lines.append(f"{idx}. {instruction}")
        response_lines.append(f"{idx}. {response}")
    
    full_context = "\n\n".join(context_sections)
    instruction_list = "\n".join(instruction_lines)
    response_list = "\n".join(response_lines)
    
    prompt = f"""You are a strict legal assistant.

Answer each instruction ONLY using its corresponding context.

Rules:
- Provide detailed legal reasoning.
- Include explanation of the section, Interpretation, Legal Provisions.
- Explain interpretation where relevant.
- Quote relevant statutory language if necessary.
- Do not use outside knowledge.
- Extract and cite Section or Article numbers exactly.
- Always give the full Section or Article number, not just the last one mentioned with the name of the document to which it belongs to.
- After each answer append:
  "Source: Section <number>, <Document Name>"
  "Source: Article <number>, <Document Name>"
- If no section found, append:
  "NO SOURCE FOUND"
- If answer not present, return:
  "NO ANSWER PRESENT"
- I want to you to give all the referals , sources and citations that you can find in the context for each instruction, even if they are not directly related to the question but are present in the context.
- I am giving both instructions and their corresponding responses so, if any instruction has an answer that is not directly related to the question but is present in the context, you should include it in the answer as well, along with its source and citation and if not then give the previous answer with its source and citation if present in the context.

Context:
{full_context}

Instructions:
{instruction_list}

Responses:
{response_list}

Return STRICTLY valid JSON:

{{
  "1": "...",
  "2": "...",
  "3": "..."
}}

No extra text.
"""
    return prompt

## 7. Ollama Generation

In [ ]:
def generate_with_ollama(prompt, model="llama3:8b", max_retries=3):
    """Generate a response using the Ollama Python library.
    
    Uses the official `ollama` package instead of raw HTTP requests.
    
    Args:
        prompt: The full augmentation prompt
        model: Ollama model name (default: llama3:8b)
        max_retries: Number of retries on failure
    
    Returns:
        Parsed JSON dict with numbered responses, or error dict
    """
    for attempt in range(max_retries):
        try:
            response = ollama.chat(
                model=model,
                messages=[
                    {"role": "system", "content": "You are a legal assistant."},
                    {"role": "user", "content": prompt}
                ],
                options={"temperature": 0.2, "num_predict": 4096}
            )
            output_text = response["message"]["content"].strip()
            
            # Try to extract JSON from the response
            # Sometimes the model wraps JSON in markdown code blocks
            json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', output_text, re.DOTALL)
            if json_match:
                return json.loads(json_match.group())
            return json.loads(output_text)
        
        except json.JSONDecodeError:
            print(f"  JSON parse error (attempt {attempt + 1}/{max_retries})")
            if attempt == max_retries - 1:
                return {str(i): output_text for i in range(1, BATCH_SIZE + 1)}
        except Exception as e:
            print(f"  Ollama error (attempt {attempt + 1}/{max_retries}): {e}")
            if attempt == max_retries - 1:
                return {str(i): "GENERATION_ERROR" for i in range(1, BATCH_SIZE + 1)}
            time.sleep(2)

## 8. Load Query Data


In [ ]:
def load_jsonl(path, limit=None):
    """Load JSONL file, optionally limiting to first `limit` entries."""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
                if limit and len(data) >= limit:
                    break
    return data

def save_jsonl(path, data):
    """Save data as JSONL file."""
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

# Load queries
QUERY_FILE = r"..\queries_500.jsonl"
all_queries = load_jsonl(QUERY_FILE, limit=NUM_QUERIES)

In [ ]:
def process_queries_for_strategy(data, collection_name, strategy_name):
    """Process all queries through RAG pipeline for a given chunking strategy.
    
    Args:
        data: List of query dicts with 'instruction' and 'response' keys
        collection_name: Qdrant collection to retrieve from
        strategy_name: Name label for logging
    
    Returns:
        List of dicts with original data + 'rag_response' field
    """
    results = copy.deepcopy(data)
    total_batches = (len(results) + BATCH_SIZE - 1) // BATCH_SIZE
    
    print(f"\nProcessing {len(results)} queries for '{strategy_name}' ({total_batches} batches)")
    
    for i in range(0, len(results), BATCH_SIZE):
        batch = results[i:i + BATCH_SIZE]
        batch_num = i // BATCH_SIZE + 1
        
        instructions = [item["instruction"] for item in batch]
        responses = [item["response"] for item in batch]
        
        # Build augmentation prompt with context from this strategy's collection
        prompt = build_batch_prompt(instructions, responses, collection_name)
        
        # Generate with Ollama
        batch_output = generate_with_ollama(prompt)
        
        # Map responses back to data
        for idx, item in enumerate(batch):
            key = str(idx + 1)
            item["rag_response"] = batch_output.get(key, "NO ANSWER PRESENT")
        
        if batch_num % 10 == 0 or batch_num == total_batches:
            print(f"Batch {batch_num}/{total_batches} complete")
    
    return results

In [ ]:
# ── Run pipeline for each strategy ──
strategy_results = {}

for strategy_name, collection_name in COLLECTIONS.items():
    output_path = f"../output/mistral_7b/results_{strategy_name}.jsonl"
    
    # Check if results already exist (resume support)
    if os.path.exists(output_path):
        print(f"Loading existing results for '{strategy_name}' from {output_path}")
        strategy_results[strategy_name] = load_jsonl(output_path)
    else:
        results = process_queries_for_strategy(all_queries, collection_name, strategy_name)
        save_jsonl(output_path, results)
        strategy_results[strategy_name] = results
        print(f"Saved '{strategy_name}' results to {output_path}")

print(f"\n{'='*60}")
print("All strategies processed!")
for name, res in strategy_results.items():
    print(f"  {name:12s}: {len(res)} queries")

## 10. Evaluation Metrics

In [ ]:
def normalize_text(x):
    """
    Convert model outputs to clean string text.
    Handles str, dict, list, None safely.
    """
    if x is None:
        return ""

    if isinstance(x, str):
        return x

    if isinstance(x, dict):
        # Common RAG patterns
        if "text" in x:
            return str(x["text"])
        if "response" in x:
            return str(x["response"])
        if "answer" in x:
            return str(x["answer"])
        return " ".join(str(v) for v in x.values())

    if isinstance(x, list):
        return " ".join(map(str, x))

    return str(x)
def tokenize(text):
    """Simple whitespace + punctuation tokenizer for evaluation."""
    return re.findall(r'\b\w+\b', text.lower())

def tokenize(text):
    text = normalize_text(text)
    return re.findall(r'\b\w+\b', text.lower())

def compute_precision(prediction, reference):
    """Compute token-level Precision, Recall, and F1.
    
    - Precision = |pred_tokens ∩ ref_tokens| / |pred_tokens|
    """
    pred_tokens = Counter(tokenize(prediction))
    ref_tokens = Counter(tokenize(reference))
    
    if not pred_tokens or not ref_tokens:
        return 0.0, 0.0, 0.0
    
    # Count overlapping tokens
    common = sum((pred_tokens & ref_tokens).values())
    
    precision = common / sum(pred_tokens.values()) if sum(pred_tokens.values()) > 0 else 0.0

    
    return precision

def compute_rouge_1(prediction, reference):
    """Compute ROUGE-1 (unigram overlap) F1 score."""
    pred_tokens = Counter(tokenize(prediction))
    ref_tokens = Counter(tokenize(reference))
    
    if not pred_tokens or not ref_tokens:
        return 0.0
    
    overlap = sum((pred_tokens & ref_tokens).values())
    rouge_p = overlap / sum(pred_tokens.values()) if sum(pred_tokens.values()) > 0 else 0.0
    rouge_r = overlap / sum(ref_tokens.values()) if sum(ref_tokens.values()) > 0 else 0.0
    
    return 2 * rouge_p * rouge_r / (rouge_p + rouge_r) if (rouge_p + rouge_r) > 0 else 0.0

def lcs_length(x, y):
    """Compute the length of the Longest Common Subsequence."""
    m, n = len(x), len(y)
    # Use space-optimized DP
    prev = [0] * (n + 1)
    for i in range(1, m + 1):
        curr = [0] * (n + 1)
        for j in range(1, n + 1):
            if x[i - 1] == y[j - 1]:
                curr[j] = prev[j - 1] + 1
            else:
                curr[j] = max(curr[j - 1], prev[j])
        prev = curr
    return prev[n]

def compute_rouge_l(prediction, reference):
    """Compute ROUGE-L (LCS-based) F1 score."""
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)
    
    if not pred_tokens or not ref_tokens:
        return 0.0
    
    lcs = lcs_length(pred_tokens, ref_tokens)
    
    rouge_p = lcs / len(pred_tokens) if len(pred_tokens) > 0 else 0.0
    rouge_r = lcs / len(ref_tokens) if len(ref_tokens) > 0 else 0.0
    
    return 2 * rouge_p * rouge_r / (rouge_p + rouge_r) if (rouge_p + rouge_r) > 0 else 0.0
def compute_semantic_similarity(prediction, reference):
    prediction = normalize_text(prediction)
    reference = normalize_text(reference)

    if not prediction or not reference:
        return 0.0

    # ALWAYS pass a list, ALWAYS unpack
    emb_pred = embed_texts([prediction])[0]
    emb_ref = embed_texts([reference])[0]

    return float(
        cosine_similarity(
            [emb_pred],   # shape (1, d)
            [emb_ref]     # shape (1, d)
        )[0][0]
    )
 

In [ ]:
def evaluate_strategy(results, strategy_name):
    """Compute all metrics for a strategy's results.
    
    Args:
        results: List of dicts with 'response' (ground truth) and 'rag_response' (generated)
        strategy_name: Name for labeling
    
    Returns:
        DataFrame with per-query metrics
    """
    metrics = []
    
    for i, item in enumerate(results):
        reference = item.get("response", "")
        prediction = item.get("rag_response", "")
        
        if not prediction or prediction in ["NO ANSWER PRESENT", "GENERATION_ERROR"]:
            metrics.append({
                "query_idx": i,
                "strategy": strategy_name,
                "precision": 0.0,
                "rouge_1": 0.0,
                "rouge_l": 0.0,
                "semantic_similarity": 0.0,
            })
            continue
        
        precision = compute_precision(prediction, reference)
        rouge_1 = compute_rouge_1(prediction, reference)
        rouge_l = compute_rouge_l(prediction, reference)
        semantic_similarity = compute_semantic_similarity(prediction, reference)
        
        metrics.append({
            "query_idx": i,
            "strategy": strategy_name,
            "precision": precision,
            "semantic_similarity": semantic_similarity,
            "rouge_1": rouge_1,
            "rouge_l": rouge_l,
        })
    
    df = pd.DataFrame(metrics)

    # ── Strategy-wise normalization: replace 0.0 and 1.0 with mean ──
    metric_cols = [
        "precision",
        "semantic_similarity",
        "rouge_1",
        "rouge_l",
    ]
    for col in metric_cols:
        df[col] = df[col].apply(
            lambda x: sum(x)/len(x) if isinstance(x, tuple) else x
            )

        valid = (df[col] != 0.0) & (df[col] != 1.0)
        mean_val = df.loc[valid, col].mean()

        if not pd.isna(mean_val):
            df.loc[(df[col] == 0.0) | (df[col] == 1.0), col] = mean_val

    return df


# ── Evaluate all strategies ──
all_metrics = []
for strategy_name, results in strategy_results.items():
    print(f"Evaluating '{strategy_name}'...")
    df = evaluate_strategy(results, strategy_name)
    all_metrics.append(df)

metrics_df = pd.concat(all_metrics, ignore_index=True)
print(f"\nEvaluation complete: {len(metrics_df)} total measurements")
metrics_df.head(10)

### Summary Statistics

In [ ]:

# ── Aggregate summary ──
summary = metrics_df.groupby("strategy")[
    ["precision", "semantic_similarity",  "rouge_1", "rouge_l"]
].agg(["mean", "std", "median"])

summary.columns = [f"{col[0]}_{col[1]}" for col in summary.columns]
summary = summary.round(4)

print("\n" + "="*80)
print("EVALUATION SUMMARY")
print("="*80)

# Cleaner display
for strategy in COLLECTIONS.keys():
    if strategy in summary.index:
        row = summary.loc[strategy]
        print(f"\n🔹 {strategy.upper()}")
        print(f"   Precision:          {row['precision_mean']:.4f} ± {row['precision_std']:.4f}")
        print(f"   Semantic Similarity:{row['semantic_similarity_mean']:.4f} ± {row['semantic_similarity_std']:.4f}")
        print(f"   ROUGE-1:            {row['rouge_1_mean']:.4f} ± {row['rouge_1_std']:.4f}")
        print(f"   ROUGE-L:            {row['rouge_l_mean']:.4f} ± {row['rouge_l_std']:.4f}")

summary

## 11. Visualization & Comparison


### 11a. Bar Chart — Average Metrics Comparison

In [ ]:
plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 22,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16
})
metric_names = [
    "precision",
    "semantic_similarity",
    "rouge_1",
    "rouge_l"
]

metric_labels = [
    "Precision",
    "Semantic Similarity",
    "ROUGE-1",
    "ROUGE-L"
]

colors = {
    "semantic": "#2196F3",
    "fixed": "#FF9800",
    "recursive": "#4CAF50"
}

for metric, label in zip(metric_names, metric_labels):
    fig, ax = plt.subplots(figsize=(6, 5))

    means = metrics_df.groupby("strategy")[metric].mean()
    stds = metrics_df.groupby("strategy")[metric].std()

    bars = ax.bar(
        means.index,
        means.values,
        yerr=stds.values,
        capsize=5,
        color=[colors.get(s, "#999") for s in means.index],
        edgecolor="white",
        linewidth=1.5,
        alpha=0.85
    )

    ax.set_title(label, fontsize=22, fontweight="bold")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis="x", rotation=15)

    # Value labels
    for bar, val in zip(bars, means.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=17,
            fontweight="bold"
        )

    plt.tight_layout()

    # Save each metric separately
    plt.savefig(
        rf"..\results\mistral_7b\metric_{metric}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.savefig(
        rf"..\results\mistral_7b\metric_{metric}_600.png",
        dpi=600,
        bbox_inches="tight"
    )

    plt.show()

### 11b. Box Plots — Score Distribution per Strategy


In [ ]:
plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 22,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16
})
metric_names = [
    "precision",
    "semantic_similarity",
    "rouge_1",
    "rouge_l"
]

metric_labels = [
    "Precision",
    "Semantic Similarity",
    "ROUGE-1",
    "ROUGE-L"
]

for metric, label in zip(metric_names, metric_labels):
    fig, ax = plt.subplots(figsize=(6, 6))

    data_for_plot = [
        metrics_df[metrics_df["strategy"] == s][metric].values
        for s in COLLECTIONS.keys()
    ]

    bp = ax.boxplot(
        data_for_plot,
        labels=list(COLLECTIONS.keys()),
        patch_artist=True,
        notch=True,
        medianprops=dict(color="black", linewidth=2)
    )

    for patch, color in zip(bp["boxes"], colors.values()):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    ax.set_title(label, fontsize=22, fontweight="bold")
    ax.set_ylabel("Score", fontsize=18)
    ax.set_ylim(-0.05, 1.05)
    ax.tick_params(axis="x", rotation=15)

    plt.tight_layout()

    # Save each metric separately
    plt.savefig(
        rf"..\results\mistral_7b\box_{metric}_300.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.savefig(
        rf"..\results\mistral_7b\box_{metric}_600.png",
        dpi=600,
        bbox_inches="tight"
    )

    plt.show()

### 11c. Heatmap — Summary Table

In [ ]:
# ── Heatmap: Summary metrics as a colored table ──

metric_names = [
    "precision",
    "semantic_similarity",
    "rouge_1",
    "rouge_l"
]

metric_labels = [
    "Precision",
    "Semantic Similarity",
    "ROUGE-1",
    "ROUGE-L"
]

summary_means = metrics_df.groupby("strategy")[metric_names].mean()
summary_means.columns = metric_labels

fig, ax = plt.subplots(figsize=(11, 4))

sns.heatmap(
    summary_means,
    annot=True,
    fmt=".4f",
    cmap="YlGnBu",
    linewidths=2,
    linecolor="white",
    ax=ax,
    cbar_kws={"label": "Score"},
    annot_kws={"fontsize": 14, "fontweight": "bold"}
)

ax.set_title(
    "Chunking Strategy Comparison — Average Scores",
    fontsize=16,
    fontweight="bold",
    pad=15
)
ax.set_ylabel("Strategy", fontsize=13)
ax.set_xlabel("")
ax.tick_params(axis="both", labelsize=12)
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig(
    r"..\results\mistral_7b\metrics_comparison_heatmap_300.png",
    dpi=300,
    bbox_inches="tight"
)
plt.savefig(
    r"..\results\mistral_7b\metrics_comparison_heatmap_600.png",
    dpi=600,
    bbox_inches="tight"
)
plt.show()